# Product Portfolio Segmentation using K-Means Clustering

This notebook covers the complete machine learning lifecycle for clustering products into strategic segments based on their **Price** and **Monthly Sales Volume**:
1. **Exploratory Data Analysis & Feature Engineering** (connecting to MongoDB)
2. **Feature Standardization** (using StandardScaler)
3. **Model Selection & Evaluation** (Elbow method SSE & Silhouette Coefficient plots)
4. **K-Means Model Training & Centroid Mapping**
5. **Cluster Scatter Plot Visualization**
6. **Database Persistence**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

### Step 1: Connect to MongoDB & Load Product / Order Data

In [ ]:
# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["smart_supply_chain"]

# Fetch products
products = list(db["products"].find())
df_products = pd.DataFrame(products)
print(f"Loaded {len(df_products)} products from collection.")

# Fetch order lines to calculate monthly volume
order_lines = list(db["order_lines"].find())
df_lines = pd.DataFrame(order_lines)
print(f"Loaded {len(df_lines)} order lines from collection.")

### Step 2: Feature Engineering
We aggregate sales quantities per product, scale them by 1.5 to establish monthly volume baselines, and merge this feature into the products dataframe.

In [ ]:
# Calculate monthly volume (quantity * 1.5)
df_volume = df_lines.groupby("product_id")["quantity"].sum().reset_index()
df_volume["monthly_volume"] = df_volume["quantity"] * 1.5

# Merge volume with products
df_data = pd.merge(df_products, df_volume, left_on="sku", right_on="product_id", how="left")
df_data["monthly_volume"] = df_data["monthly_volume"].fillna(0.0)
df_data = df_data[["sku", "name", "price", "monthly_volume"]]
df_data.head(10)

### Step 3: Feature Scaling (Standardization)
Since K-Means is a distance-based clustering algorithm, features must be on the same scale (mean=0, variance=1) to prevent the feature with larger absolute values (Price) from dominating.

In [ ]:
features = ["price", "monthly_volume"]
X = df_data[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Sample scaled coordinates (mean=0, std=1):")
print(X_scaled[:5])

### Step 4: Model Evaluation (Elbow & Silhouette Analysis)
We evaluate cluster numbers from $K=1$ to $K=8$ using two metrics:
1. **Inertia (SSE)**: Within-cluster sum of squares (lower is better, look for an 'elbow').
2. **Silhouette Score**: Measures cluster separation and cohesion (range -1 to 1, higher is better).

In [ ]:
sse = []
silhouette_scores = []
k_range = range(2, 9)

# Run K=1 for Elbow base reference
sse.append(KMeans(n_clusters=1, random_state=42, n_init=10).fit(X_scaled).inertia_)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    sse.append(kmeans.inertia_)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)
    print(f"K = {k} | SSE (Inertia) = {kmeans.inertia_:.2f} | Silhouette Score = {score:.4f}")

In [ ]:
# Plot Elbow and Silhouette curves side-by-side
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# SSE / Elbow Plot
ax[0].plot(range(1, 9), sse, marker='o', color='#8b5cf6', linewidth=2.5)
ax[0].set_title('Elbow Method (SSE)', fontsize=14, fontweight='bold', pad=10)
ax[0].set_xlabel('Number of Clusters (K)', fontsize=12)
ax[0].set_ylabel('Inertia / SSE', fontsize=12)
ax[0].grid(True, linestyle='--', alpha=0.5)

# Silhouette Plot
ax[1].plot(k_range, silhouette_scores, marker='o', color='#3b82f6', linewidth=2.5)
ax[1].set_title('Silhouette Coefficient Curve', fontsize=14, fontweight='bold', pad=10)
ax[1].set_xlabel('Number of Clusters (K)', fontsize=12)
ax[1].set_ylabel('Silhouette Score', fontsize=12)
ax[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Step 5: Final Model Training & Centroid Interpretation
We train K-Means with $K=3$ and dynamically map the cluster IDs to named portfolio tiers (`HIGH VALUE`, `VOLUME DRIVERS`, and `LOW PERFORMERS`) based on centroid characteristics.

In [ ]:
# Train final model
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)
df_data["cluster_id"] = cluster_labels

# Get and inverse transform centroids
centroids = kmeans_final.cluster_centers_
unscaled_centroids = scaler.inverse_transform(centroids)

cluster_mapping = {}
for i, (price, vol) in enumerate(unscaled_centroids):
    if price > 800:
        cluster_mapping[i] = "HIGH VALUE"
    elif vol > 400:
        cluster_mapping[i] = "VOLUME DRIVERS"
    else:
        cluster_mapping[i] = "LOW PERFORMERS"
        
df_data["cluster"] = df_data["cluster_id"].map(cluster_mapping)

print("Final Cluster Mapping Results:")
for cid, tier in cluster_mapping.items():
    print(f"Cluster {cid} -> {tier} (Centroid: Price=${unscaled_centroids[cid][0]:.2f}, Volume={unscaled_centroids[cid][1]:.1f})")

### Step 6: Cluster Visualization
We generate a scatter plot showing all products color-coded by their cluster assignments, with the cluster centroids displayed as large markers.

In [ ]:
plt.figure(figsize=(12, 7))
colors = {
    "HIGH VALUE": "#3b82f6",      # Premium Blue
    "VOLUME DRIVERS": "#8b5cf6",    # Premium Purple
    "LOW PERFORMERS": "#9ca3af"     # Premium Gray
}

# Plot product scatter points
sns.scatterplot(
    data=df_data, 
    x="price", 
    y="monthly_volume", 
    hue="cluster", 
    palette=colors,
    s=110, 
    alpha=0.85, 
    edgecolor='black', 
    linewidth=0.7
)

# Plot centroids
for cid, tier in cluster_mapping.items():
    cx, cy = unscaled_centroids[cid]
    plt.scatter(
        cx, cy, 
        color=colors[tier], 
        marker='X', 
        s=350, 
        edgecolor='black', 
        linewidth=2.5, 
        label=f"{tier} Centroid"
)

plt.title("Product Segmentation via K-Means Clustering (K=3)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Product Price ($)", fontsize=12)
plt.ylabel("Monthly Sales Volume", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.35)
plt.legend(frameon=True, facecolor='white', framealpha=0.9, edgecolor='none', fontsize=11)
plt.show()

### Step 7: Summary Analysis Statistics

In [ ]:
summary = df_data.groupby("cluster").agg(
    count=("sku", "count"),
    avg_price=("price", "mean"),
    avg_volume=("monthly_volume", "mean")
).reset_index()
summary

### Step 8: Persist Results to MongoDB
We show how the cluster assignments are updated back into the MongoDB documents for the FastAPI backend and Angular frontend to consume.

In [ ]:
from pymongo import UpdateOne

bulk_operations = []
for _, row in df_data.iterrows():
    bulk_operations.append(
        UpdateOne(
            {"sku": int(row["sku"])},
            {"$set": {
                "monthly_volume": float(row["monthly_volume"]),
                "cluster": str(row["cluster"])
            }}
        )
    )

if bulk_operations:
    result = db["products"].bulk_write(bulk_operations)
    print(f"Database update summary: Matched={result.matched_count}, Modified={result.modified_count}")